# 03 — Results Analysis
Load and analyze experiment results, generate charts and tables for the report.

In [ ]:
import json
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

RESULTS_DIR = '../results'

In [ ]:
# Load the latest results
results_files = [f for f in os.listdir(RESULTS_DIR) if f.endswith('_results.json')]
print(f'Found {len(results_files)} result files:', results_files)

if results_files:
    latest = sorted(results_files)[-1]
    with open(os.path.join(RESULTS_DIR, latest), 'r') as f:
        results = json.load(f)
    print(f'Loaded: {latest}')
    print(f'Total problems: {results["total_problems"]}')
    print(f'Overall accuracy: {results["overall_accuracy"]:.4f}')

In [ ]:
# Build summary table
if results:
    summary = results.get('summary', {})
    rows = []
    for key, val in summary.items():
        rows.append({
            'Model': val.get('model', ''),
            'Method': val.get('method', ''),
            'Dataset': val.get('dataset', ''),
            'Accuracy': val.get('accuracy', 0),
            'Correct': val.get('correct', 0),
            'Total': val.get('total', 0),
        })
    df = pd.DataFrame(rows)
    display(df.style.format({'Accuracy': '{:.2%}'}))

In [ ]:
# Accuracy heatmap
if results:
    pivot = df.pivot_table(index='Model', columns='Method', values='Accuracy') * 100
    fig = go.Figure(data=go.Heatmap(
        z=pivot.values.round(1),
        x=list(pivot.columns),
        y=list(pivot.index),
        colorscale='RdYlGn',
        text=pivot.values.round(1),
        texttemplate='%{text}%',
        zmin=0, zmax=100,
    ))
    fig.update_layout(title='Accuracy Heatmap (Model x Method)', height=400)
    fig.show()

In [ ]:
# Bar chart comparison
if results:
    fig = px.bar(df, x='Model', y='Accuracy', color='Method', barmode='group',
                 title='Accuracy by Model and Prompt Method')
    fig.update_yaxes(tickformat='.0%')
    fig.show()